# GIMMS EOS process models


### Setup


In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import dual_annealing
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=RuntimeWarning)

EVAL_DIR = os.path.abspath(os.getcwd())
if os.path.basename(EVAL_DIR) != "evaluation":
    cand = "/Users/xingyihuang/Jupyter Code/Phenology/New Code/model/evaluation"
    if os.path.isdir(cand):
        EVAL_DIR = cand
os.chdir(EVAL_DIR)

BASE = os.path.abspath(os.path.join(EVAL_DIR, "..", ".."))
TMP_DIR = os.path.join(BASE, "results/model/evaluation/gimms")
DATA_DIR = os.path.join(TMP_DIR, "data")
CACHE_DIR = os.path.join(TMP_DIR, "cache")
FIG_DIR = os.path.join(TMP_DIR, "figures")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

CACHE_EXTRACT = os.path.join(DATA_DIR, "extract_coarse_from_fine41_clim_1982_2022.npz")
CACHE_SIAMG_PREP_CSV = os.path.join(DATA_DIR, "siamg_preseason_p_coarse_from_fig2_L.csv")
CACHE_SIAMG_PREP_NPZ = os.path.join(DATA_DIR, "siamg_preseason_p_coarse_from_fig2_L.npz")
CACHE_RESULTS = os.path.join(CACHE_DIR, "eval_results_boundexp_siam_kge08_cv5.csv")
FIG_COMPARE = os.path.join(FIG_DIR, "eval_model_compare_boundexp_siam_kge08_cv5.png")
FORCE_REFIT = False

YEARS = list(range(1982, 2023))
CHILL_START_DOY = 173
MODEL_ORDER = ["CDD", "CDDP", "SIAM", "SIAMP"]

N_PILOT = 2106
MAXITER = 100
N_FOLDS = 5
SEED = 42

print("EVAL_DIR:", EVAL_DIR)
print("TMP_DIR:", TMP_DIR)
print("YEARS:", YEARS[0], "-", YEARS[-1])
print("CHILL_START_DOY:", CHILL_START_DOY)
print("Models:", MODEL_ORDER)
print("MAXITER:", MAXITER, "| N_FOLDS:", N_FOLDS, "| FORCE_REFIT:", FORCE_REFIT)


### Daylength


In [ ]:
import numpy as np

def day_length(day_of_year, latitude):
    day_of_year = np.array(day_of_year)
    latitude_rad = np.deg2rad(latitude)
    P = np.arcsin(0.39795 * np.cos(0.2163108 + 2 * np.arctan(0.9671396 * np.tan(0.00860 * (day_of_year - 186)))))
    numerator = np.sin(np.deg2rad(0.8333)) + np.sin(latitude_rad) * np.sin(P)
    denominator = np.cos(latitude_rad) * np.cos(P)
    acos_arg = numerator / denominator
    acos_arg = np.clip(acos_arg, -1.0, 1.0)
    day_light_hours = 24 - (24 / np.pi) * np.arccos(acos_arg)
    return day_light_hours


### Models


In [ ]:
F_HI_CDDP = 2500.0

F_HI_SIAMP = 3000.0

BP_MAX_CDDP = 10.0

BP_MAX_SIAMP = 10.0

CP_MAX_CDDP = 5.0

CP_MAX_SIAMP = 5.0

B_MAX_SIAM = 30.0

B_MAX_SIAMP = 30.0

def get_model_list():

    return {

        "CDD": dict(fun=CDD_model, needs_sos=False, needs_tp=False,

                    lower=[0.0, 0.0], upper=[25.0, F_HI_CDDP]),

        "CDDP": dict(fun=CDDP_model, needs_sos=False, needs_tp=True,

                     lower=[0.0, 0.0, -BP_MAX_CDDP, -CP_MAX_CDDP],

                     upper=[25.0, F_HI_CDDP, BP_MAX_CDDP, CP_MAX_CDDP]),

        "SIAM": dict(fun=SIAM_model, needs_sos=True, needs_tp=False,

                     lower=[0.0, 8.0, 0.1, -B_MAX_SIAM],

                     upper=[25.0, 16.0, F_HI_SIAMP, B_MAX_SIAM]),

        "SIAMP": dict(fun=SIAMP_model, needs_sos=True, needs_tp=True,

                      lower=[0.0, 8.0, 0.1, -B_MAX_SIAMP, -BP_MAX_SIAMP, -CP_MAX_SIAMP],

                      upper=[25.0, 16.0, F_HI_SIAMP, B_MAX_SIAMP, BP_MAX_SIAMP, CP_MAX_SIAMP]),

    }


In [ ]:
import numpy as np

def _rate_matrix(Tmini, hours, T_base, P_base=None):

    temp = np.asarray(Tmini, float)

    rate = np.maximum(float(T_base) - temp, 0.0)

    if P_base is not None:

        h = np.asarray(hours, float)[:, None]

        rate = rate * np.maximum(1.0 - h / float(P_base), 0.0)

    rate[:CHILL_START_DOY, :] = 0.0

    rate[~np.isfinite(rate)] = 0.0

    return rate

def _eos_from_rate(rate, qcrit):

    qcrit = np.asarray(qcrit, float)

    if qcrit.ndim == 0:

        qcrit = np.full(rate.shape[1], float(qcrit))

    bad = ~np.isfinite(qcrit) | (qcrit <= 0)

    cum = np.cumsum(rate, axis=0)

    hit = cum >= qcrit[None, :]

    any_hit = np.any(hit, axis=0) & ~bad

    out = np.full(rate.shape[1], np.nan)

    out[any_hit] = np.argmax(hit[:, any_hit], axis=0) + 1.0

    return out

def _sos_anom(predictor, data):

    return np.asarray(predictor, dtype=float).ravel()

def _quad_P_threshold(F_base, Pz, b_p, c_p):

    """Wu et al. 2022-style: Ycrit = F + b_p*P_z + c_p*P_z^2 (clip > 0)."""

    Pz = np.asarray(Pz, float).ravel()

    q = np.asarray(F_base, float) + float(b_p) * Pz + float(c_p) * (Pz ** 2)

    return np.maximum(q, 1.0)

def CDD_model(par, data):

    Tb, F = map(float, par[:2])

    return _eos_from_rate(_rate_matrix(data["Tmini"], data["hours"], Tb), F)

def CDDP_model(par, data):

    """CDD + quadratic preseason P on F_crit (Wu-style CDDP).
    Params: Tb, F, b_p, c_p
    F_y = F + b_p*P_z + c_p*P_z^2
    """

    Tb, F, b_p, c_p = map(float, par[:4])

    qcrit = _quad_P_threshold(F, data["P_z"], b_p, c_p)

    return _eos_from_rate(_rate_matrix(data["Tmini"], data["hours"], Tb), qcrit)

def SIAM_model(par, predictor, data):

    Tb, Pb, F, b = map(float, par[:4])

    sos = _sos_anom(predictor, data)

    qcrit = F + b * sos

    return _eos_from_rate(_rate_matrix(data["Tmini"], data["hours"], Tb, Pb), qcrit)

def SIAMP_model(par, predictor, data):

    """SIAM + quadratic preseason P on F_crit (Wu-style SIAMP).
    Params: Tb, Pb, F, b_sos, b_p, c_p
    F_y = F + b_sos*SOS' + b_p*P_z + c_p*P_z^2
    """

    Tb, Pb, F, b, b_p, c_p = map(float, par[:6])

    sos = _sos_anom(predictor, data)

    F_base = float(F) + float(b) * sos

    qcrit = _quad_P_threshold(F_base, data["P_z"], b_p, c_p)

    return _eos_from_rate(_rate_matrix(data["Tmini"], data["hours"], Tb, Pb), qcrit)

print("Models:", list(get_model_list()))

for name, info in get_model_list().items():

    print(f"  {name} bounds: lo={info['lower']}  hi={info['upper']}")


### Load coarse dataset


In [ ]:
                                                                                              

assert os.path.exists(CACHE_EXTRACT), (

    f"Missing {CACHE_EXTRACT}\nRun build_gimms_coarse_dataset.ipynb first."

)

assert os.path.exists(CACHE_SIAMG_PREP_NPZ), (

    f"Missing {CACHE_SIAMG_PREP_NPZ}\nRun build_gimms_coarse_dataset.ipynb first."

)

z = np.load(CACHE_EXTRACT)

zp = np.load(CACHE_SIAMG_PREP_NPZ)

prep_df = pd.read_csv(CACHE_SIAMG_PREP_CSV) if os.path.exists(CACHE_SIAMG_PREP_CSV) else None

eos = z["eos"]; sos = z["sos"]; daily_t = z["daily_t"]; daily_p = z["daily_p"]

mean_eos = z["mean_eos"] if "mean_eos" in z.files else np.nanmean(eos, axis=1)

lats = z["latitude"]; lons = z["longitude"]

annual_t = z["annual_t"]; annual_p = z["annual_p"]

preseason_p = zp["preseason_p"]

preseason_t = zp["preseason_t"] if "preseason_t" in zp.files else None

if preseason_t is None:

    raise RuntimeError("preseason_t missing from SIAMG prep npz (needed for T–P match gate)")

mat = zp["mat"] if "mat" in zp.files else np.nanmean(annual_t, axis=1)

mapa = zp["map"] if "map" in zp.files else np.nanmean(annual_p, axis=1)

w_spatial = zp["w"] if "w" in zp.files else None

if preseason_p.shape[0] != eos.shape[0]:

    raise RuntimeError(f"preseason_p n={preseason_p.shape[0]} != eos n={eos.shape[0]}")

print("Loaded", CACHE_EXTRACT)

print("Loaded", CACHE_SIAMG_PREP_NPZ)

print("n coarse cells:", eos.shape[0],

      "| years:", int(z["years"][0]), "-", int(z["years"][-1]),

      "| n_years:", eos.shape[1])

if "n_fine_per_coarse" in z.files:

    print("fine pixels / coarse cell: mean={:.1f}, min={}, max={}".format(

        float(z["n_fine_per_coarse"].mean()),

        int(z["n_fine_per_coarse"].min()),

        int(z["n_fine_per_coarse"].max()),

    ))

print("preseason_t/p:", preseason_t.shape, preseason_p.shape,

      "| w mean={:.3f} range=[{:.3f},{:.3f}]".format(

          float(np.nanmean(w_spatial)), float(np.nanmin(w_spatial)), float(np.nanmax(w_spatial))))

if prep_df is not None:

    cols = [c for c in ["preseason_t_months", "preseason_p_months", "mat", "map", "w"] if c in prep_df.columns]

    print(prep_df[cols].describe().round(3))


### Fit / CV


In [ ]:
                                                    

PILOT_SEED = 42

def pack_data(daily_t_pix, lat, year_idx, P_z=None, T_z=None, sos_baseline=None):

    Tmini = daily_t_pix[:, year_idx].copy()

    Tmini[~np.isfinite(Tmini)] = 999.0

    n_days = Tmini.shape[0]

    hours = np.asarray(day_length(np.arange(1, n_days + 1, dtype=float), lat), float).ravel()

    Li = hours[:, None] * np.ones((1, len(year_idx)))

    data = {"T": Tmini, "Tmini": Tmini, "hours": hours, "Li": Li}

    if P_z is not None:

        data["P_z"] = np.asarray(P_z, float)[year_idx]

    if T_z is not None:

        data["T_z"] = np.asarray(T_z, float)[year_idx]

    if sos_baseline is not None:

        data["sos_baseline"] = float(sos_baseline)

    return data

def standardize_from_train(values, train_indices):

    x = np.asarray(values, float)

    mean = float(np.nanmean(x[train_indices]))

    sd = float(np.nanstd(x[train_indices]))

    if not np.isfinite(sd) or sd < 1e-9:

        sd = 1.0

    return (x - mean) / sd

def kge(obs, sim):

    ok = np.isfinite(obs) & np.isfinite(sim)

    o, s = obs[ok], sim[ok]

    if o.size < 3 or np.std(o) < 1e-12 or np.std(s) < 1e-12:

        return np.nan

    r = float(np.corrcoef(s, o)[0, 1])

    alpha = float(np.std(s) / np.std(o))

    beta = float(np.mean(s) / np.mean(o))

    return float(1.0 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))

def predict(minfo, par, sos_vals, data):

    if minfo["needs_sos"]:

        return minfo["fun"](par, sos_vals, data)

    return minfo["fun"](par, data)

def eval_metrics(obs, sim):

    ok = np.isfinite(obs) & np.isfinite(sim)

    if ok.sum() < 3:

        return dict(rmse=np.nan, r=np.nan, kge=np.nan)

    o, s = obs[ok], sim[ok]

    rmse = float(np.sqrt(np.mean((s - o) ** 2)))

    r = float(np.corrcoef(s, o)[0, 1]) if (np.std(o) > 1e-12 and np.std(s) > 1e-12) else np.nan

    return dict(rmse=rmse, r=r, kge=kge(obs, sim))

def joint_cost(par, minfo, data, obs, sos_vals):

    if not np.all(np.isfinite(par)):

        return 1e10

    try:

        pred = predict(minfo, par, sos_vals, data)

    except Exception:

        return 1e10

    m = eval_metrics(obs, pred)

    if not np.isfinite(m["rmse"]):

        return 1e10

    return float(m["rmse"])

model_list = get_model_list()

model_list = {k: v for k, v in model_list.items() if k in MODEL_ORDER}

RESULTS_CSV = CACHE_RESULTS

def _print_eval_summary(res):

    summary = res.groupby("model")[["rmse", "r", "kge"]].mean().reindex(MODEL_ORDER)

    print(summary.round(3))

    for base, gate in [

        ("CDD", "CDDP"),

        ("SIAM", "SIAMP"),

    ]:

        b = res[res["model"] == base].set_index("pixel")

        g = res[res["model"] == gate].set_index("pixel")

        if b.empty or g.empty:

            continue

        d = g[["rmse", "r", "kge"]] - b[["rmse", "r", "kge"]]

        print(

            f"{base} -> {gate}: ΔRMSE={d['rmse'].mean():+.3f} (median {d['rmse'].median():+.3f}), "

            f"Δr={d['r'].mean():+.3f}, ΔKGE={d['kge'].mean():+.3f}, "

            f"wins rmse={(d['rmse']<0).sum()}/{len(d)}"

        )

    for a, bname in [("CDD", "SIAM"), ("CDDP", "SIAMP")]:

        A = res[res["model"] == a].set_index("pixel")

        B = res[res["model"] == bname].set_index("pixel")

        d = B[["rmse", "r", "kge"]] - A[["rmse", "r", "kge"]]

        print(

            f"{a} -> {bname}: ΔRMSE={d['rmse'].mean():+.3f}, "

            f"Δr={d['r'].mean():+.3f}, ΔKGE={d['kge'].mean():+.3f}, "

            f"wins rmse={(d['rmse']<0).sum()}/{len(d)}"

        )

if (not FORCE_REFIT) and os.path.exists(RESULTS_CSV):

    print(f"Loading cached evaluation results: {RESULTS_CSV}")

    res = pd.read_csv(RESULTS_CSV)

    print(f"Loaded {res['pixel'].nunique()} pixels × {res['model'].nunique()} models (skip fit)")

    _print_eval_summary(res)

else:

    model_list = get_model_list()

    model_list = {k: v for k, v in model_list.items() if k in MODEL_ORDER}

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    n_pix, ny = eos.shape

    rows = []

    t0 = time.time()

    eligible = []

    for i in range(n_pix):

        n_ok = int((

            np.isfinite(eos[i]) & np.isfinite(sos[i]) & np.isfinite(preseason_p[i])

            & np.isfinite(preseason_t[i]) & np.isfinite(daily_t[i]).any(axis=0)

        ).sum())

        if n_ok >= max(10, N_FOLDS * 2):

            eligible.append(i)

    eligible = np.asarray(eligible, dtype=int)

    if N_PILOT is None:

        run_idx = eligible

    else:

        rng = np.random.default_rng(PILOT_SEED)

        take = min(int(N_PILOT), len(eligible))

        run_idx = np.sort(rng.choice(eligible, size=take, replace=False))

    print(

        f"Fitting {len(run_idx)} / {len(eligible)} | MAXITER={MAXITER} | N_FOLDS={N_FOLDS}"

    )

    for j, pi in enumerate(run_idx):

        valid = np.where(

            np.isfinite(eos[pi]) & np.isfinite(sos[pi]) & np.isfinite(preseason_p[pi])

            & np.isfinite(preseason_t[pi]) & np.isfinite(daily_t[pi]).any(axis=0)

        )[0]

        print(f"Cell {j+1}/{len(run_idx)} idx={pi} (n={valid.size})...", flush=True)

        oof_pred = {m: np.full(ny, np.nan) for m in model_list}

        for tri, tei in kf.split(valid):

            tr, te = valid[tri], valid[tei]

            p_z = standardize_from_train(preseason_p[pi], tr)

            t_z = standardize_from_train(preseason_t[pi], tr)

            sos_z = standardize_from_train(sos[pi], tr)

            obs_tr, sos_tr = eos[pi, tr], sos_z[tr]

            for mname in [m for m in MODEL_ORDER if m in model_list]:

                minfo = model_list[mname]

                dtr = pack_data(

                    daily_t[pi], lats[pi], tr,

                    P_z=p_z if minfo["needs_tp"] else None,

                    T_z=t_z if minfo["needs_tp"] else None,

                )

                dte = pack_data(

                    daily_t[pi], lats[pi], te,

                    P_z=p_z if minfo["needs_tp"] else None,

                    T_z=t_z if minfo["needs_tp"] else None,

                )

                res_opt = dual_annealing(

                    lambda par, _m=minfo, _d=dtr, _o=obs_tr, _s=sos_tr: joint_cost(par, _m, _d, _o, _s),

                    bounds=list(zip(minfo["lower"], minfo["upper"])),

                    maxiter=MAXITER,

                    seed=SEED,

                )

                res_x = np.asarray(res_opt.x, float)

                try:

                    pred = predict(minfo, res_x, sos_z[te], dte)

                except Exception:

                    pred = np.full(te.size, np.nan)

                oof_pred[mname][te] = pred

        for mname in model_list:

            m = eval_metrics(eos[pi], oof_pred[mname])

            rows.append({

                "pixel": int(pi), "latitude": float(lats[pi]), "longitude": float(lons[pi]),

                "model": mname, "rmse": m["rmse"], "r": m["r"], "kge": m["kge"],

            })

        print(f"  elapsed {time.time()-t0:.1f}s", flush=True)

    res = pd.DataFrame(rows)

    out_res = os.path.join(

        CACHE_DIR,

        f"pilot_results_boundexp_siam_kge08_coarse{len(run_idx)}_cv{N_FOLDS}.csv",

    )

    res.to_csv(out_res, index=False)

    res.to_csv(CACHE_RESULTS, index=False)

    print("Saved", out_res)

    print("Saved", CACHE_RESULTS)

    _print_eval_summary(res)

res.head()


### 5. Compare models


In [ ]:
                                               
from scipy import stats
from matplotlib.ticker import MultipleLocator
if "res" not in globals() or res is None or (hasattr(res, "empty") and res.empty):
    if not os.path.exists(CACHE_RESULTS):
        raise FileNotFoundError(
            f"Missing {CACHE_RESULTS}. Run the Fit/CV cell with FORCE_REFIT=True once."
        )
    res = pd.read_csv(CACHE_RESULTS)
    print("Loaded", CACHE_RESULTS, f"({res['pixel'].nunique()} pixels)")
model_colors = {
    "CDD": "#a3d4e1",
    "CDDP": "#5cadd8",
    "SIAM": "#ea9e87",
    "SIAMP": "#cf847e",
}
order_pref = list(MODEL_ORDER)
order = [m for m in order_pref if m in set(res["model"])]
n_pix = len(res["pixel"].unique())
PAIRS = [("CDD", "CDDP"), ("SIAM", "SIAMP")]
def paired_ttest_p(base, pmod, metric):
    a = res.loc[res["model"] == base].set_index("pixel")[metric]
    b = res.loc[res["model"] == pmod].set_index("pixel")[metric]
    common = a.index.intersection(b.index)
    if len(common) < 3:
        return np.nan
    av = a.loc[common].to_numpy(dtype=float)
    bv = b.loc[common].to_numpy(dtype=float)
    ok = np.isfinite(av) & np.isfinite(bv)
    if ok.sum() < 3 or np.nanstd(av[ok] - bv[ok]) == 0:
        return np.nan
    _, pval = stats.ttest_rel(av[ok], bv[ok])                        
    return float(pval)
def stars(p):
    if not np.isfinite(p):
        return "ns"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"
def plot_metric_bars(col, ylabel, ylim, y_tick):
    """One separate bar figure for a metric (slightly wider)."""
    mean_vals, err_vals = [], []
    for m in order:
        vals = res.loc[res["model"] == m, col].dropna()
        mean_vals.append(float(vals.mean()) if not vals.empty else np.nan)
        n = int(len(vals))
        std_val = float(vals.std(ddof=1)) if n > 1 else 0.0
        se = std_val / np.sqrt(n) if n > 1 and np.isfinite(std_val) else 0.0
        err_vals.append(2.0 * se)
    y_lo0, y_hi0 = ylim
    y_span0 = y_hi0 - y_lo0
    fig, ax = plt.subplots(figsize=(6, 5))                               
    bars = ax.bar(
        order, mean_vals,
        color=[model_colors.get(m, "#cccccc") for m in order],
    )
    xpos = {m: float(i) for i, m in enumerate(order)}
    label_y = {}
    offset = 0.02 * y_span0
    for bar, m, mean_val, err in zip(bars, order, mean_vals, err_vals):
        x = xpos[m]
        if not np.isfinite(mean_val):
            continue
        y_top = mean_val
        if err > 0:
            half_cap = bar.get_width() * 0.12
            ax.plot([x, x], [mean_val, mean_val + err], color="#333333", lw=0.9, solid_capstyle="butt", zorder=3)
            ax.plot(
                [x - half_cap, x + half_cap], [mean_val + err, mean_val + err],
                color="#333333", lw=0.9, solid_capstyle="butt", zorder=3,
            )
            y_top = mean_val + err
        ypos = y_top + offset
        label_val = np.floor(mean_val * 100.0) / 100.0
        ax.text(
            x, ypos, f"{label_val:.2f}",
            ha="center", va="bottom", fontsize=16, fontweight="bold",
        )
        label_y[m] = ypos
    step = 0.07 * y_span0
    tick = 0.015 * y_span0
    gap = 0.16 * y_span0
    y_max = y_hi0
    for k, (base, pmod) in enumerate(PAIRS):
        if base not in xpos or pmod not in xpos:
            continue
        pval = paired_ttest_p(base, pmod, col)
        s = stars(pval)
        x1, x2 = xpos[base], xpos[pmod]
        hi = max(
            label_y.get(base, mean_vals[order.index(base)]),
            label_y.get(pmod, mean_vals[order.index(pmod)]),
        )
        y = hi + gap
        ax.plot([x1, x1, x2, x2], [y - tick, y, y, y - tick],
                color="#333333", lw=0.7, clip_on=False)
        ax.text(
            0.5 * (x1 + x2), y + tick * 0.25, s,
            ha="center", va="bottom", fontsize=20, color="#222222",
        )
        print(f"{col} {base} vs {pmod}: p={pval:.2e} → {s}")
        y_max = max(y_max, y + 1.2 * step)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=45, ha="right", fontsize=16)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_ylim(y_lo0, y_max)
    ax.yaxis.set_major_locator(MultipleLocator(y_tick))
    ax.grid(False)
    ax.tick_params(axis="both", labelsize=16)
    fig.subplots_adjust(bottom=0.22, top=0.90)
    return fig
fig_rmse = plot_metric_bars("rmse", "RMSE (days)", ylim=(8.0, 11.0), y_tick=1)
fig_r = plot_metric_bars("r", "r", ylim=(0.15, 0.35), y_tick=0.1)
fig_kge = plot_metric_bars("kge", "KGE", ylim=(0.05, 0.25), y_tick=0.1)
base_compare = FIG_COMPARE if "FIG_COMPARE" in globals() else os.path.join(
    FIG_DIR if "FIG_DIR" in globals() else TMP_DIR,
    f"pilot_model_compare_test_fig4style_coarse{n_pix}_cv{N_FOLDS}.png",
)
fig_rmse_path = base_compare.replace(".png", "_rmse.png")
fig_r_path = base_compare.replace(".png", "_r.png")
fig_kge_path = base_compare.replace(".png", "_kge.png")
fig_rmse.savefig(fig_rmse_path, dpi=500, bbox_inches="tight")
fig_r.savefig(fig_r_path, dpi=500, bbox_inches="tight")
fig_kge.savefig(fig_kge_path, dpi=500, bbox_inches="tight")
plt.show()
print("Saved", fig_rmse_path)
print("Saved", fig_r_path)
print("Saved", fig_kge_path)
print("Brackets: paired two-tailed Student t (Fig. 3 style); * p<0.05, ** p<0.01, ns otherwise")
res.groupby("model")[["rmse", "r", "kge"]].mean().reindex(order).round(3)
